In [1]:
# raw Docling document object 
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
result = converter.convert("CELEX_02013R0575-20250629_EN_TXT.pdf")
docling_doc = result.document

print(f"📄 Docling document has {len(docling_doc.pages)} pages")
print(f"📄 You can use docling_doc.iterate_items() to extract content")

2025-12-24 15:41:33,089 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-12-24 15:41:33,222 - INFO - Going to convert document batch...
2025-12-24 15:41:33,223 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 44ae89a68fc272bc7889292e9b5a1bad
2025-12-24 15:41:34,124 - WARNING - The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
2025-12-24 15:41:34,125 - INFO - Loading plugin 'docling_defaults'
2025-12-24 15:41:34,129 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-12-24 15:41:34,156 - WARNING - The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
2025-12-24 15:41:34,158 - INFO - Loading plugin 'docling_defaults'
2025-12-24 15:41:34,168 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-12-24 15:41:34,557 - INFO - Accelerator device: 'cuda:0'
[INFO] 2025-12-2

📄 Docling document has 897 pages
📄 You can use docling_doc.iterate_items() to extract content


In [2]:
# Check what items exist and their properties
headings_found = []
clanak_found = []
all_text_samples = []

count = 0
for item, level in docling_doc.iterate_items():
    count += 1
    
    # Get text content from the item
    txt = ""
    if hasattr(item, 'text') and item.text:
        txt = item.text.strip()
    elif hasattr(item, 'caption') and item.caption:
        txt = item.caption.strip()
    elif hasattr(item, 'title') and item.title:
        txt = item.title.strip()
    
    # Check if it has a label
    if hasattr(item, 'label'):
        print(f"Item {count}: Type={type(item).__name__}, Label='{item.label}', Text='{txt[:100]}...'")
        
        # Collect headings
        if item.label == "heading":
            headings_found.append(txt)
            
        # Check for Article or article specifically
        if txt and ("article" in txt.lower() or "Article" in txt):
            clanak_found.append(f"Label: {item.label}, Text: {txt}")
    
    # Collect some text samples
    if txt and len(all_text_samples) < 20:
        all_text_samples.append(f"Type: {type(item).__name__}, Label: {getattr(item, 'label', 'None')}, Text: {txt[:100]}")
    
    if count > 100:  # Limit output
        break

print(f"\n=== SUMMARY ===")
print(f"Total items processed: {count}")
print(f"Headings found: {len(headings_found)}")
print(f"Items mentioning 'Article': {len(clanak_found)}")


print(f"\n=== HEADINGS FOUND ===")
for i, heading in enumerate(headings_found[:100]):  # Show first 10
    print(f"{i+1}: {heading}")

print(f"\n=== ARTICLE MENTIONS ===")
for article in clanak_found[:100]:  # Show first 10
    print(article)

print(f"\n=== SAMPLE TEXT CONTENT ===")
for sample in all_text_samples[:100]:  # Show first 10
    
    print(sample)
    

Item 1: Type=TextItem, Label='text', Text='This  text  is  meant  purely  as  a  documentation  tool  and  has  no  legal  effect.  The  Union'...'
Item 2: Type=SectionHeaderItem, Label='section_header', Text='► B ► M9  REGULATION  (EU)  No  575/2013  OF  THE  EUROPEAN  PARLIAMENT AND OF THE COUNCIL...'
Item 3: Type=TextItem, Label='text', Text='of  26  June  2013...'
Item 4: Type=TextItem, Label='text', Text='on prudential requirements  for credit institutions  and amending Regulation (EU) No 648/2012 ◄...'
Item 5: Type=FormulaItem, Label='formula', Text='...'
Item 6: Type=SectionHeaderItem, Label='section_header', Text='Amended  by:...'
Item 7: Type=TableItem, Label='table', Text='...'
Item 8: Type=TextItem, Label='text', Text='Council  of  31  March  2021...'
Item 9: Type=TableItem, Label='table', Text='...'
Item 10: Type=SectionHeaderItem, Label='section_header', Text='Corrected  by:...'
Item 11: Type=ListItem, Label='list_item', Text='C1 Corrigendum,  OJ  L  208,  2.8.2013,  p.  6

In [3]:
# Extract articles from the document
import re
from langchain_core.documents import Document

chunks = []
current = None

print("=== EXTRACTING ARTICLES (Croatian: Članak) ===\n")

# Pattern for Croatian articles: "Članak 123" or "Article 123" (for multilingual docs)
article_pattern = re.compile(r'^(Članak|Article)\s+(\d+)', re.IGNORECASE)

for item, level in docling_doc.iterate_items():
    # Get text content from the item
    txt = ""
    if hasattr(item, 'text') and item.text:
        txt = item.text.strip()
    elif hasattr(item, 'caption') and item.caption:
        txt = item.caption.strip()
    elif hasattr(item, 'title') and item.title:
        txt = item.title.strip()
    
    if not txt:
        continue
    
    # Check if this is an article heading
    article_match = article_pattern.match(txt)
    
    if article_match:
        article_type = article_match.group(1)  # "Članak" or "Article"
        article_num = article_match.group(2)   # The number
        
        print(f"✓ Found {article_type} {article_num}: {txt[:80]}...")
        
        # Save previous chunk before starting a new one
        if current:
            chunks.append(Document(**current))
        
        # Get page number
        try:
            page_num = item.prov[0].page_no if hasattr(item, 'prov') and item.prov else 1
        except:
            page_num = 1
        
        # Start new article chunk
        current = {
            "page_content": txt + "\n",
            "metadata": {
                "type": "article",
                "article_no": f"{article_type} {article_num}",
                "article_number": int(article_num),
                "page": page_num,
                "item_type": type(item).__name__,
                "label": getattr(item, 'label', 'unknown'),
                "language": "hr" if article_type == "Članak" else "en"
            }
        }
    elif current:
        # Append content to current article
        current["page_content"] += txt + "\n"

# Don't forget the last chunk
if current:
    chunks.append(Document(**current))

# Results summary
print(f"\n{'='*60}")
print(f"✅ EXTRACTION COMPLETE")
print(f"{'='*60}")
print(f"Total articles extracted: {len(chunks)}")

if chunks:
    # Show distribution
    article_numbers = [c.metadata.get('article_number', 0) for c in chunks]
    print(f"Article range: {min(article_numbers)} - {max(article_numbers)}")
    
    # Show samples
    print(f"\n📋 First 5 articles:")
    for i, chunk in enumerate(chunks[:5], 1):
        article_no = chunk.metadata.get('article_no', 'Unknown')
        page = chunk.metadata.get('page', 'N/A')
        content_len = len(chunk.page_content)
        print(f"  {i}. {article_no} (Page {page}) - {content_len} chars")
        print(f"     Preview: {chunk.page_content[:100].replace(chr(10), ' ')}...")
else:
    print("⚠️  No articles found! Check the pattern or document content.")
    print("💡 Run cell 2 again to see what text patterns exist in the document.")

=== EXTRACTING ARTICLES (Croatian: Članak) ===

✓ Found Article 1: Article  1...
✓ Found Article 2: Article  2...
✓ Found Article 3: Article  3...
✓ Found Article 4: Article  4...
✓ Found Article 5: Article  5...
✓ Found Article 5: Article  5a...
✓ Found Article 6: Article  6...
✓ Found Article 7: Article  7...
✓ Found Article 8: Article  8...
✓ Found Article 9: Article  9...
✓ Found Article 10: Article  10...
✓ Found Article 10: Article  10a...
✓ Found Article 11: Article  11...
✓ Found Article 12: Article  12a...
✓ Found Article 13: Article  13...
✓ Found Article 14: Article  14...
✓ Found Article 18: Article  18...
✓ Found Article 19: Article  19...
✓ Found Article 20: Article  20...
✓ Found Article 21: Article  21...
✓ Found Article 23: Article  23...
✓ Found Article 24: Article  24...
✓ Found Article 25: Article  25...
✓ Found Article 26: Article  26...
✓ Found Article 27: Article  27...
✓ Found Article 28: Article  28...
✓ Found Article 29: Article  29...
✓ Found Article 30: Arti

In [4]:

import os
from dotenv import load_dotenv
from langchain_astradb import AstraDBVectorStore
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# Load environment variables
load_dotenv()

print(f"✅ Successfully extracted {len(chunks)} legal document chunks with Docling!")
print(f"Sample chunk metadata: {chunks[0].metadata}")
print(f"Sample content preview: {chunks[0].page_content[:200]}...")

✅ Successfully extracted 737 legal document chunks with Docling!
Sample chunk metadata: {'type': 'article', 'article_no': 'Article 1', 'article_number': 1, 'page': 3, 'item_type': 'SectionHeaderItem', 'label': <DocItemLabel.SECTION_HEADER: 'section_header'>, 'language': 'en'}
Sample content preview: Article  1
Scope
This Regulation lays down uniform rules concerning general prudential requirements  that  institutions,  financial  holding  companies  and  mixed financial  holding  companies  super...


In [ ]:
# Initialize NVIDIA embeddings
print("🔧 Initializing NVIDIA embeddings...")
embeddings = NVIDIAEmbeddings(
    model="nvidia/nv-embedqa-e5-v5",
    api_key=os.getenv("NVIDIA_API_KEY")
)

# Connect to Astra DB
print("🗄️ Connecting to Astra DB...")
collection_name = "crr_docling_chunks"

vectorstore = AstraDBVectorStore(
    embedding=embeddings,
    collection_name=collection_name,
    token=os.getenv("ASTRA_DB_TOKEN"),
    api_endpoint=os.getenv("ASTRA_DB_API_ENDPOINT"),
)

print(f"✅ Connected to Astra DB collection: {collection_name}")

2025-12-24 15:47:19,175 - INFO - vector store default init, collection 'crr_docling_chunks'


🔧 Initializing NVIDIA embeddings...
🗄️ Connecting to Astra DB...


2025-12-24 15:47:19,850 - INFO - Attempting to fetch keyspace from environment variable 'ASTRA_DB_KEYSPACE'
2025-12-24 15:47:19,852 - INFO - Detecting API environment 'prod' from supplied endpoint
2025-12-24 15:47:20,062 - INFO - createCollection('crr_docling_chunks')
2025-12-24 15:47:23,132 - INFO - HTTP Request: POST https://50cf877b-c447-42e7-b570-8e1ca0a0dbde-us-east1.apps.astra.datastax.com/api/json/v1/default_keyspace "HTTP/1.1 200 OK"
2025-12-24 15:47:23,134 - INFO - finished createCollection('crr_docling_chunks')


✅ Connected to Astra DB collection: crr_docling_chunks


In [6]:
# Fix: Split large chunks to fit NVIDIA's 512 token limit
from langchain_text_splitters import RecursiveCharacterTextSplitter

def estimate_tokens(text):
    """Rough estimation: 1 token ≈ 4 characters"""
    return len(text) / 4

def split_large_chunks_with_redetection(chunks, max_tokens=512):
    """
    Split chunks that exceed token limit AND re-detect article numbers.
    
    CRITICAL: This fixes the metadata inheritance bug where sub-chunks
    incorrectly inherit article numbers from their parent chunk.
    """
    import re
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    
    print(f"🔧 Checking chunks for NVIDIA's {max_tokens} token limit...")
    print(f"🔍 Re-detecting article numbers in sub-chunks...")
    
    valid_chunks = []
    oversized_count = 0
    redetected_count = 0
    
    # Article pattern for re-detection
    article_pattern = re.compile(r'^(Članak|Article)\s+(\d+)', re.IGNORECASE | re.MULTILINE)
    
    # Text splitter for oversized chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1800,  # ~450 tokens (safe margin)
        chunk_overlap=200,
        separators=[
            "\n\n",          # Paragraph breaks (DON'T split at Article)
            "\n",            # Line breaks
            ". ",            # Sentence breaks
            " ",             # Word breaks
            ""
        ],
        keep_separator=True,
        length_function=len,
    )
    
    def estimate_tokens(text):
        """Rough estimation: 1 token ≈ 4 characters"""
        return len(text) / 4
    
    for i, chunk in enumerate(chunks):
        estimated_tokens = estimate_tokens(chunk.page_content)
        
        if estimated_tokens <= max_tokens:
            # Chunk is fine as-is
            valid_chunks.append(chunk)
        else:
            # Chunk is too large, split it
            oversized_count += 1
            original_article = chunk.metadata.get('article_number', 'unknown')
            print(f"  Splitting chunk {i+1} (Article {original_article}, {estimated_tokens:.0f} tokens)")
            
            # Split the oversized chunk
            sub_chunks = text_splitter.split_documents([chunk])
            
            # Process each sub-chunk
            for j, sub_chunk in enumerate(sub_chunks):
                # Start with original metadata
                sub_chunk.metadata = chunk.metadata.copy()
                sub_chunk.metadata['sub_chunk'] = j + 1
                sub_chunk.metadata['total_sub_chunks'] = len(sub_chunks)
                sub_chunk.metadata['original_chunk_tokens'] = int(estimated_tokens)
                sub_chunk.metadata['original_article_number'] = original_article
                
                # ✅ RE-DETECT article number in this specific sub-chunk
                first_300_chars = sub_chunk.page_content[:300].strip()
                article_match = article_pattern.search(first_300_chars)
                
                if article_match:
                    # Found an article in this sub-chunk - update metadata
                    article_type = article_match.group(1)
                    article_num = article_match.group(2)
                    detected_article_number = int(article_num)
                    
                    # Only update if different from original
                    if detected_article_number != original_article:
                        sub_chunk.metadata['article_no'] = f"{article_type} {article_num}"
                        sub_chunk.metadata['article_number'] = detected_article_number
                        sub_chunk.metadata['language'] = "hr" if article_type == "Članak" else "en"
                        sub_chunk.metadata['article_redetected'] = True
                        redetected_count += 1
                        print(f"    ✅ Re-detected {article_type} {article_num} in sub-chunk {j+1} (was {original_article})")
                    else:
                        sub_chunk.metadata['article_redetected'] = False
                else:
                    # No article detected - this is a continuation chunk
                    sub_chunk.metadata['article_redetected'] = False
                    sub_chunk.metadata['is_continuation'] = True
                
                # Verify sub-chunk size
                sub_tokens = estimate_tokens(sub_chunk.page_content)
                if sub_tokens <= max_tokens:
                    valid_chunks.append(sub_chunk)
                else:
                    print(f"    ⚠️  Warning: Sub-chunk still too large ({sub_tokens:.0f} tokens)")
                    # Could recursively split again here if needed
    
    print(f"\n✅ Processed {len(chunks)} chunks:")
    print(f"   - {len(chunks) - oversized_count} chunks were within limit")
    print(f"   - {oversized_count} chunks were split into sub-chunks")
    print(f"   - {redetected_count} sub-chunks had articles re-detected")
    print(f"   - Final total: {len(valid_chunks)} chunks")
    
    # Verify article coverage
    article_numbers = sorted(set(c.metadata.get('article_number') for c in valid_chunks if c.metadata.get('article_number')))
    if article_numbers:
        print(f"\n📊 Article coverage:")
        print(f"   - Range: Article {min(article_numbers)} - {max(article_numbers)}")
        print(f"   - Total unique articles: {len(article_numbers)}")
        
        # Check for specific articles
        if 32 in article_numbers and 33 in article_numbers:
            print(f"   - ✅ Articles 32 & 33 are present!")
        else:
            if 32 not in article_numbers:
                print(f"   - ❌ Article 32 is MISSING!")
            if 33 not in article_numbers:
                print(f"   - ❌ Article 33 is MISSING!")
    
    return valid_chunks

# Split the chunks
print("Preparing chunks for NVIDIA embeddings...")
valid_chunks = split_large_chunks_with_redetection(chunks, max_tokens=512)

Preparing chunks for NVIDIA embeddings...
🔧 Checking chunks for NVIDIA's 512 token limit...
🔍 Re-detecting article numbers in sub-chunks...
  Splitting chunk 3 (Article 3, 532 tokens)
  Splitting chunk 4 (Article 4, 17592 tokens)
  Splitting chunk 5 (Article 5, 1142 tokens)
  Splitting chunk 6 (Article 5, 958 tokens)
  Splitting chunk 7 (Article 6, 1134 tokens)
  Splitting chunk 8 (Article 7, 743 tokens)
  Splitting chunk 9 (Article 8, 1349 tokens)
  Splitting chunk 13 (Article 11, 1364 tokens)
  Splitting chunk 17 (Article 18, 1674 tokens)
  Splitting chunk 19 (Article 20, 1684 tokens)
  Splitting chunk 20 (Article 21, 1353 tokens)
  Splitting chunk 24 (Article 26, 1314 tokens)
  Splitting chunk 26 (Article 28, 2475 tokens)
  Splitting chunk 27 (Article 29, 831 tokens)
  Splitting chunk 31 (Article 33, 568 tokens)
  Splitting chunk 34 (Article 36, 2289 tokens)
  Splitting chunk 37 (Article 39, 529 tokens)
  Splitting chunk 44 (Article 46, 948 tokens)
  Splitting chunk 46 (Article 47, 

In [7]:
# Now add the properly sized chunks to Astra DB
print(f"📚 Adding {len(valid_chunks)} properly-sized chunks to Astra DB...")

# Process in smaller batches for stability
batch_size = 25  # Smaller batches
total_batches = (len(valid_chunks) + batch_size - 1) // batch_size

successfully_added = 0
failed_chunks = []

for i in range(0, len(valid_chunks), batch_size):
    batch = valid_chunks[i:i + batch_size]
    batch_num = i // batch_size + 1
    
    print(f"Processing batch {batch_num}/{total_batches} ({len(batch)} chunks)...")
    
    try:
        # Check batch for token limits before adding
        for chunk in batch:
            tokens = estimate_tokens(chunk.page_content)
            if tokens > 512:
                print(f"  Warning: Chunk still has {tokens:.0f} tokens")
        
        vectorstore.add_documents(batch)
        successfully_added += len(batch)
        print(f"  ✅ Batch {batch_num} added successfully")
        
    except Exception as e:
        print(f"  ❌ Batch {batch_num} failed: {str(e)[:100]}")
        failed_chunks.extend(batch)

print(f"\n📊 Results:")
print(f"✅ Successfully added: {successfully_added} chunks")
print(f"❌ Failed: {len(failed_chunks)} chunks")

if failed_chunks:
    print(f"\nDebugging first failed chunk:")
    failed_chunk = failed_chunks[0]
    print(f"Content length: {len(failed_chunk.page_content)}")
    print(f"Estimated tokens: {estimate_tokens(failed_chunk.page_content):.0f}")
    print(f"Content preview: {failed_chunk.page_content[:200]}...")

📚 Adding 1618 properly-sized chunks to Astra DB...
Processing batch 1/65 (25 chunks)...


2025-12-24 15:47:47,552 - INFO - inserting 25 documents in 'crr_docling_chunks'
2025-12-24 15:47:47,554 - INFO - insertMany(chunk) on 'crr_docling_chunks'
2025-12-24 15:47:48,844 - INFO - HTTP Request: POST https://50cf877b-c447-42e7-b570-8e1ca0a0dbde-us-east1.apps.astra.datastax.com/api/json/v1/default_keyspace/crr_docling_chunks "HTTP/1.1 200 OK"
2025-12-24 15:47:48,846 - INFO - finished insertMany(chunk) on 'crr_docling_chunks'
2025-12-24 15:47:48,850 - INFO - finished inserting 25 documents in 'crr_docling_chunks'


  ✅ Batch 1 added successfully
Processing batch 2/65 (25 chunks)...


2025-12-24 15:47:49,444 - INFO - inserting 25 documents in 'crr_docling_chunks'
2025-12-24 15:47:49,446 - INFO - insertMany(chunk) on 'crr_docling_chunks'
2025-12-24 15:47:50,201 - INFO - HTTP Request: POST https://50cf877b-c447-42e7-b570-8e1ca0a0dbde-us-east1.apps.astra.datastax.com/api/json/v1/default_keyspace/crr_docling_chunks "HTTP/1.1 200 OK"
2025-12-24 15:47:50,203 - INFO - finished insertMany(chunk) on 'crr_docling_chunks'
2025-12-24 15:47:50,205 - INFO - finished inserting 25 documents in 'crr_docling_chunks'


  ✅ Batch 2 added successfully
Processing batch 3/65 (25 chunks)...


...

  ✅ Batch 63 added successfully
Processing batch 64/65 (25 chunks)...


2025-12-24 15:49:32,342 - INFO - inserting 25 documents in 'crr_docling_chunks'
2025-12-24 15:49:32,344 - INFO - insertMany(chunk) on 'crr_docling_chunks'
2025-12-24 15:49:35,142 - INFO - HTTP Request: POST https://50cf877b-c447-42e7-b570-8e1ca0a0dbde-us-east1.apps.astra.datastax.com/api/json/v1/default_keyspace/crr_docling_chunks "HTTP/1.1 200 OK"
2025-12-24 15:49:35,145 - INFO - finished insertMany(chunk) on 'crr_docling_chunks'
2025-12-24 15:49:35,147 - INFO - finished inserting 25 documents in 'crr_docling_chunks'


  ✅ Batch 64 added successfully
Processing batch 65/65 (18 chunks)...


2025-12-24 15:49:35,790 - INFO - inserting 18 documents in 'crr_docling_chunks'
2025-12-24 15:49:35,792 - INFO - insertMany(chunk) on 'crr_docling_chunks'
2025-12-24 15:49:37,399 - INFO - HTTP Request: POST https://50cf877b-c447-42e7-b570-8e1ca0a0dbde-us-east1.apps.astra.datastax.com/api/json/v1/default_keyspace/crr_docling_chunks "HTTP/1.1 200 OK"
2025-12-24 15:49:37,402 - INFO - finished insertMany(chunk) on 'crr_docling_chunks'
2025-12-24 15:49:37,403 - INFO - finished inserting 18 documents in 'crr_docling_chunks'


  ✅ Batch 65 added successfully

📊 Results:
✅ Successfully added: 1618 chunks
❌ Failed: 0 chunks


In [8]:
# Save valid_chunks as pickle file
import pickle
from datetime import datetime

# Create filename with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
pickle_filename = f"legal_chunks_docling_{timestamp}.pkl"

print(f"💾 Saving {len(valid_chunks)} chunks to pickle file...")

# Save the chunks
with open(pickle_filename, 'wb') as f:
    pickle.dump(valid_chunks, f)

print(f"✅ Chunks saved to: {pickle_filename}")
print(f"📊 File contains {len(valid_chunks)} processed legal document chunks")

# Also save metadata summary
metadata_summary = {
    'total_chunks': len(valid_chunks),
    'source_document': 'CELEX_02013R0575-20250629_EN_TXT.pdf',
    'processing_date': datetime.now().isoformat(),
    'chunk_types': [chunk.metadata.get('type', 'unknown') for chunk in valid_chunks],
    'articles_found': list(set([chunk.metadata.get('article_no', 'Unknown') for chunk in valid_chunks])),
    'processing_method': 'docling_with_nvidia_token_splitting'
}

summary_filename = f"legal_chunks_summary_{timestamp}.pkl"
with open(summary_filename, 'wb') as f:
    pickle.dump(metadata_summary, f)

print(f"📋 Metadata summary saved to: {summary_filename}")

💾 Saving 1618 chunks to pickle file...
✅ Chunks saved to: legal_chunks_docling_20251224_154949.pkl
📊 File contains 1618 processed legal document chunks
📋 Metadata summary saved to: legal_chunks_summary_20251224_154949.pkl


In [9]:
# Initialize Google Gemini for legal analysis
print("🤖 Initializing Google Gemini for legal document analysis...")
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1,  # Low temperature for accuracy in legal context
    google_api_key=os.getenv("GEMINI_API_KEY")
)

# Create legal-specific prompt template
legal_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a specialized legal document assistant with expertise in financial regulations, particularly the Capital Requirements Regulation (CRR). 

Your role is to:
1. Provide accurate, precise answers based solely on the provided legal document context
2. Always cite specific articles, sections, or provisions when referencing information
3. Distinguish between mandatory requirements ("shall", "must") and optional provisions ("may", "should")
4. Explain complex legal concepts in clear, professional language
5. When uncertain, clearly state limitations and suggest consulting legal counsel

Important guidelines:
- Only use information from the provided context
- Never speculate or provide general legal advice
- Always reference specific article numbers when applicable
- Maintain professional, formal tone appropriate for legal documentation"""),
    
    ("user", """Based on the following legal document excerpts, please answer the question:

Context: {context}

Question: {question}

Please provide a comprehensive answer with specific references to articles and provisions.""")
])

# Create the legal RAG chain
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    """Format retrieved documents for the prompt"""
    formatted = []
    for doc in docs:
        article_info = doc.metadata.get('article_no', 'Unknown Article')
        page_info = doc.metadata.get('page', 'Unknown Page')
        content = doc.page_content.strip()
        formatted.append(f"[{article_info}, Page {page_info}]\n{content}")
    return "\n\n---\n\n".join(formatted)

# Create retriever
print("🔍 Setting up legal document retriever...")
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8}  # Retrieve top 8 most relevant chunks
)

# Build the complete RAG chain
legal_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | legal_prompt
    | llm
    | StrOutputParser()
)

print("✅ Legal RAG system is ready!")
print("\n🎯 System Summary:")
print(f"📄 Document: CELEX_02013R0575-20250629_EN_TXT.pdf (Capital Requirements Regulation)")
print(f"📚 Chunks stored: {successfully_added} legal document sections")
print(f"🧠 Embeddings: NVIDIA nv-embedqa-e5-v5")
print(f"🗄️ Vector Store: Astra DB")
print(f"🤖 LLM: Google Gemini 2.0 Flash")
print(f"📖 Specialized for: Financial regulation legal analysis")

🤖 Initializing Google Gemini for legal document analysis...
🔍 Setting up legal document retriever...
✅ Legal RAG system is ready!

🎯 System Summary:
📄 Document: CELEX_02013R0575-20250629_EN_TXT.pdf (Capital Requirements Regulation)
📚 Chunks stored: 1618 legal document sections
🧠 Embeddings: NVIDIA nv-embedqa-e5-v5
🗄️ Vector Store: Astra DB
🤖 LLM: Google Gemini 2.0 Flash
📖 Specialized for: Financial regulation legal analysis


In [10]:
# Test your legal RAG system
def query_legal_document(question):
    """Query the legal RAG system"""
    try:
        print(f"🔍 Searching for: {question}")
        print("=" * 50)
        
        # Get answer from RAG chain
        response = legal_rag_chain.invoke(question)
        
        print("📋 Legal Analysis:")
        print(response)
        print("=" * 50)
        
        return response
    except Exception as e:
        print(f"❌ Error querying system: {e}")
        return None

# Example legal queries you can try:
sample_queries = [
    "What are the capital requirements for credit institutions under Article 92?",
    "What is the definition of Common Equity Tier 1 capital?",
    "What are the requirements for large exposures?",
    "How are credit risk adjustments calculated?",
    "What are the liquidity coverage requirements?"
]

print("🎯 Sample Legal Queries:")
for i, query in enumerate(sample_queries, 1):
    print(f"{i}. {query}")

print("\n💡 Try querying your system:")
print("response = query_legal_document('Your question about the regulation')")

🎯 Sample Legal Queries:
1. What are the capital requirements for credit institutions under Article 92?
2. What is the definition of Common Equity Tier 1 capital?
3. What are the requirements for large exposures?
4. How are credit risk adjustments calculated?
5. What are the liquidity coverage requirements?

💡 Try querying your system:
response = query_legal_document('Your question about the regulation')


In [11]:
query_legal_document("What are the requirements for large exposures?") 

🔍 Searching for: What are the requirements for large exposures?


2025-12-24 15:50:15,590 - INFO - cursor fetching a page: (empty page state) from crr_docling_chunks
2025-12-24 15:50:16,284 - INFO - HTTP Request: POST https://50cf877b-c447-42e7-b570-8e1ca0a0dbde-us-east1.apps.astra.datastax.com/api/json/v1/default_keyspace/crr_docling_chunks "HTTP/1.1 200 OK"
2025-12-24 15:50:16,287 - INFO - cursor finished fetching a page: (empty page state) from crr_docling_chunks


📋 Legal Analysis:
Based on the provided legal document excerpts, the requirements for large exposures are as follows:

1.  **General Monitoring and Control**: Institutions **shall** monitor and control their large exposures (Article 387, Page 618).

2.  **Definition of a Large Exposure**: An institution's exposure to a client or a group of connected clients **shall** be considered a large exposure where its value is equal to or exceeds 10% of its Tier 1 capital (Article 392, Page 621).

3.  **Capacity to Identify and Manage Large Exposures**: An institution **shall** have sound administrative and accounting procedures and adequate internal control mechanisms for identifying, managing, monitoring, reporting, and recording all large exposures and subsequent changes to them, in accordance with this Regulation (Article 393, Page 622).

4.  **Limits to Large Exposures**:
    *   An institution **shall not** incur an exposure to a client or group of connected clients the value of which excee

"Based on the provided legal document excerpts, the requirements for large exposures are as follows:\n\n1.  **General Monitoring and Control**: Institutions **shall** monitor and control their large exposures (Article 387, Page 618).\n\n2.  **Definition of a Large Exposure**: An institution's exposure to a client or a group of connected clients **shall** be considered a large exposure where its value is equal to or exceeds 10% of its Tier 1 capital (Article 392, Page 621).\n\n3.  **Capacity to Identify and Manage Large Exposures**: An institution **shall** have sound administrative and accounting procedures and adequate internal control mechanisms for identifying, managing, monitoring, reporting, and recording all large exposures and subsequent changes to them, in accordance with this Regulation (Article 393, Page 622).\n\n4.  **Limits to Large Exposures**:\n    *   An institution **shall not** incur an exposure to a client or group of connected clients the value of which exceeds 25% o